# Widgets

[ipywidgets](https://ipywidgets.readthedocs.io) are controls, such as sliders, dropdowns and
buttons, that live in the notebook and stay connected to Python objects in the kernel. Change one
and the other follows.

## A widget is a Python object

In [ ]:
import ipywidgets as widgets

slider = widgets.IntSlider(value=5, min=0, max=10, description="Count")
slider

In [ ]:
slider.value

In [ ]:
slider.value = 8
slider.max = 20

Move the slider and run `slider.value` again. The same object is displayed in the page and held
by the kernel.

## A gallery of controls

In [ ]:
widgets.VBox([
    widgets.FloatSlider(description="Float", value=0.5, max=1.0, step=0.01),
    widgets.IntRangeSlider(description="Range", value=[20, 60]),
    widgets.Dropdown(description="Species", options=["Adelie", "Chinstrap", "Gentoo"]),
    widgets.ToggleButtons(description="Sex", options=["female", "male"]),
    widgets.Checkbox(description="Include missing"),
    widgets.Text(description="Name", placeholder="Type here"),
    widgets.Textarea(description="Notes"),
    widgets.DatePicker(description="Date"),
    widgets.ColorPicker(description="Colour", value="#0f766e"),
    widgets.IntProgress(description="Progress", value=7, max=10),
])

## Reacting to changes

`observe` calls a function whenever a property changes. Here, the Celsius slider updates the
Fahrenheit label.

In [ ]:
celsius = widgets.FloatSlider(description="°C", min=-40, max=100, value=20)
fahrenheit = widgets.Label()

def convert(change):
    fahrenheit.value = f"{change['new'] * 9 / 5 + 32:.1f} °F"

celsius.observe(convert, names="value")
convert({"new": celsius.value})
widgets.HBox([celsius, fahrenheit])

## Linking without Python

`jslink` connects two widgets in the browser, so they stay in sync with no round trip to the
kernel.

In [ ]:
play = widgets.Play(min=0, max=100, interval=50)
progress = widgets.IntSlider(max=100)
widgets.jslink((play, "value"), (progress, "value"))
widgets.HBox([play, progress])

## `interact`

`interact` builds controls from a function's arguments and reruns the function whenever they
change. A list becomes a dropdown, a tuple becomes a slider, a bool becomes a checkbox.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

@widgets.interact(frequency=(0.5, 5.0, 0.25), damping=(0.0, 1.0, 0.05), style=["-", "--", ":"], grid=True)
def wave(frequency=1.0, damping=0.2, style="-", grid=True):
    x = np.linspace(0, 10, 500)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(x, np.exp(-damping * x) * np.sin(frequency * x), style)
    ax.set_ylim(-1.1, 1.1)
    ax.grid(grid)
    plt.show()

## A small app

Combine controls, an `Output` area and a callback into a penguin explorer.

In [ ]:
import pandas as pd

penguins = pd.read_csv("../data/penguins.csv").dropna()
numeric = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

x_axis = widgets.Dropdown(options=numeric, value="flipper_length_mm", description="x")
y_axis = widgets.Dropdown(options=numeric, value="body_mass_g", description="y")
species = widgets.SelectMultiple(options=sorted(penguins["species"].unique()), value=("Adelie", "Chinstrap", "Gentoo"), description="Species")
chart = widgets.Output()
stats = widgets.HTML()

def redraw(_=None):
    shown = penguins[penguins["species"].isin(species.value)]
    stats.value = f"<b>{len(shown)}</b> penguins shown"
    with chart:
        chart.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 4))
        for name, group in shown.groupby("species"):
            ax.scatter(group[x_axis.value], group[y_axis.value], label=name, alpha=0.7)
        ax.set_xlabel(x_axis.value)
        ax.set_ylabel(y_axis.value)
        if len(shown):
            ax.legend()
        plt.show()

for control in (x_axis, y_axis, species):
    control.observe(redraw, names="value")
redraw()

widgets.HBox([widgets.VBox([x_axis, y_axis, species, stats]), chart])

## Widgets and the kernel

Widgets belong to the running kernel. Close this notebook's tab and open it again: Zasper redraws
every widget from the kernel, with its current value.

Restart the kernel, though, and the widgets are gone, because the Python objects behind them are
gone. Run the cells again to bring them back.

## Next

Continue with [07-kernels-and-execution](07-kernels-and-execution.ipynb).